In [26]:
# Model1: XGBoost model to predict Dissolved Reactive Phosphorus (DRP)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
import xgboost as xgb  # pip install xgboost


In [27]:
# Load engineered training features and join with DRP target

features_path = "../New Datasets/Combined/combined_training_engineered.csv"
water_quality_path = "../Provided Datasets/water_quality_training_dataset.csv"

combined = pd.read_csv(features_path)
water_quality = pd.read_csv(water_quality_path)

# Standardize join keys to match `combined`
water_quality_std = water_quality.rename(
    columns={
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Sample Date": "sample_date",
    }
)

# Keep only join keys + DRP target
drp_target = water_quality_std[["latitude", "longitude", "sample_date", "Dissolved Reactive Phosphorus"]]

# Inner join to align features with DRP labels
full = combined.merge(drp_target, on=["latitude", "longitude", "sample_date"], how="inner")

print("Features shape (combined):", combined.shape)
print("Water quality shape:", water_quality.shape)
print("Joined training shape:", full.shape)
full.head()


Features shape (combined): (9319, 82)
Water quality shape: (9319, 6)
Joined training shape: (9319, 83)


,latitude,longitude,sample_date,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,...,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious,Dissolved Reactive Phosphorus
0,-34.405833,19.600556,01-10-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,4703.580299,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,20.0
1,-34.405833,19.600556,02-08-2011,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1984.043203,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,10.0
2,-34.405833,19.600556,02-12-2015,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,6827.433216,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,20.0
3,-34.405833,19.600556,03-07-2013,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1466.619596,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,10.0
4,-34.405833,19.600556,03-09-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,2919.631339,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,20.0


In [28]:
# Build feature matrix X and target y for DRP, then reduce multicollinearity

# Columns to exclude from features
exclude_cols = {
    "Dissolved Reactive Phosphorus",    # target
    "Total Alkalinity",                 # do not include
    "Electrical Conductance",           # do not include
    "latitude", "longitude",           # do not include
    "sample_date",                      # string key, not a feature
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["Dissolved Reactive Phosphorus"]

print("Initial number of features:", len(base_feature_cols))

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# Expose reduced set for feature selection
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])


Initial number of features: 79
Dropped 24 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 55
Example remaining feature columns: ['gaia_changed_ever_frac', 'gaia_recent_change_5y_frac', 'gsw_change', 'gsw_occurrence', 'gsw_transitions', 'nir', 'green', 'swir16', 'NDMI', 'MNDWI']


In [29]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 95% of total importance, but ensure at least 25 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])


Total features after multicollinearity reduction: 55
Selected features after importance-based selection: 31
Top selected features: ['esa_grass_frac_1km', 'esa_water_frac_1km', 'water_perm', 'esa_forest_frac_1km', 'gsw_recurrence_mean_1km', 'esa_shrub_frac_1km', 'gsw_occurrence_mean_1km', 'gaia_recent_change_5y_frac', 'gaia_changed_ever_frac_1km', 'esa_cropland_frac_1km', 'esa_urban_frac_1km', 'gaia_recent_change_5y_frac_1km', 'esa_other_frac_1km', 'recurrence_ratio', 'gsw_occ_x_impervious']


In [30]:
# Baseline XGBoost model (for quick R² and importances)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Baseline Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Baseline Test  R²: {r2_score(y_test, y_test_pred):.3f}")

importances_base = model.feature_importances_
fi_base = pd.DataFrame({"feature": feature_cols, "importance": importances_base})
fi_base.sort_values("importance", ascending=False).head(20)


Baseline Train R²: 0.903
Baseline Test  R²: 0.724


,feature,importance
1,esa_water_frac_1km,0.143413
3,esa_forest_frac_1km,0.064680
0,esa_grass_frac_1km,0.057453
5,esa_shrub_frac_1km,0.056219
7,gaia_recent_change_5y_frac,0.056176
6,gsw_occurrence_mean_1km,0.055709
15,esa_lccs_class,0.051167
8,gaia_changed_ever_frac_1km,0.047156
4,gsw_recurrence_mean_1km,0.042910
13,recurrence_ratio,0.040070


In [ ]:
# Stratified K-Fold + Optuna hyperparameter tuning for DRP model

n_bins = 10
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-02-24 17:41:33,574] A new study created in memory with name: no-name-65310f30-2dc5-4a32-8993-702371caf761
Best trial: 0. Best value: 0.666166:   2%|▎         | 1/40 [00:01<01:06,  1.71s/it]

[I 2026-02-24 17:41:35,288] Trial 0 finished with value: 0.6661655639331145 and parameters: {'n_estimators': 263, 'max_depth': 5, 'learning_rate': 0.03407550409309202, 'subsample': 0.7781562521053547, 'colsample_bytree': 0.7430850345151654, 'min_child_weight': 3.4219380558559767, 'gamma': 3.461380491892721, 'reg_alpha': 0.7160943737810785, 'reg_lambda': 0.4903758097532358}. Best is trial 0 with value: 0.6661655639331145.


Best trial: 0. Best value: 0.666166:   5%|▌         | 2/40 [00:03<01:05,  1.73s/it]

[I 2026-02-24 17:41:37,038] Trial 1 finished with value: 0.5634361126499401 and parameters: {'n_estimators': 327, 'max_depth': 4, 'learning_rate': 0.010364157209664557, 'subsample': 0.8021775750645204, 'colsample_bytree': 0.6527740392819013, 'min_child_weight': 8.699997932011847, 'gamma': 1.4064682284854235, 'reg_alpha': 0.7415940959938586, 'reg_lambda': 0.9948669662666578}. Best is trial 0 with value: 0.6661655639331145.


Best trial: 2. Best value: 0.676859:   8%|▊         | 3/40 [00:04<01:00,  1.64s/it]

...

[I 2026-02-24 17:42:52,072] Trial 24 finished with value: 0.6930692332803065 and parameters: {'n_estimators': 677, 'max_depth': 6, 'learning_rate': 0.014951838636400825, 'subsample': 0.8547077888948883, 'colsample_bytree': 0.6739690680783739, 'min_child_weight': 6.081579304976926, 'gamma': 0.5806356389653222, 'reg_alpha': 0.6056160988641871, 'reg_lambda': 0.3238904117402046}. Best is trial 20 with value: 0.7009529500031791.


Best trial: 20. Best value: 0.700953:  65%|██████▌   | 26/40 [01:24<01:07,  4.82s/it]

[I 2026-02-24 17:42:57,802] Trial 25 finished with value: 0.6963784433202522 and parameters: {'n_estimators': 777, 'max_depth': 6, 'learning_rate': 0.01843431574791646, 'subsample': 0.743648847402121, 'colsample_bytree': 0.5494465022879835, 'min_child_weight': 7.903806193592788, 'gamma': 1.282656769739654, 'reg_alpha': 0.6189027934957942, 'reg_lambda': 0.7891383514451412}. Best is trial 20 with value: 0.7009529500031791.


Best trial: 20. Best value: 0.700953:  68%|██████▊   | 27/40 [01:28<00:58,  4.52s/it]

[I 2026-02-24 17:43:01,648] Trial 26 finished with value: 0.6875220496365394 and parameters: {'n_estimators': 628, 'max_depth': 5, 'learning_rate': 0.029390778822528853, 'subsample': 0.8351634083670301, 'colsample_bytree': 0.5789625585583447, 'min_child_weight': 5.301273696321545, 'gamma': 0.48550402235224005, 'reg_alpha': 0.49988390849298037, 'reg_lambda': 0.4783507604397451}. Best is trial 20 with value: 0.7009529500031791.


In [ ]:
# Train final DRP model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(X_train, y_train)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final DRP model Train R²: {r2_train:.3f}")
print(f"Final DRP model Test  R²: {r2_test:.3f}")

importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting DRP (tuned model):")
print(fi_tuned.head(20).to_string(index=False))

fi_tuned.head(20)


Final DRP model Train R²: 0.889
Final DRP model Test  R²: 0.721

Top 20 most important features for predicting DRP (tuned model):
                       feature  importance
            esa_water_frac_1km    0.155634
            esa_grass_frac_1km    0.081081
       gsw_occurrence_mean_1km    0.067563
           esa_forest_frac_1km    0.065639
            esa_shrub_frac_1km    0.063249
    gaia_recent_change_5y_frac    0.056431
                esa_lccs_class    0.049961
    gaia_changed_ever_frac_1km    0.045247
       gsw_recurrence_mean_1km    0.041032
         esa_cropland_frac_1km    0.038343
                    water_perm    0.033285
gaia_recent_change_5y_frac_1km    0.030975
        gaia_changed_ever_frac    0.030496
              recurrence_ratio    0.029384
            esa_other_frac_1km    0.026016
          gsw_occ_x_impervious    0.024611
            esa_urban_frac_1km    0.020132
                          soil    0.016008
                         MNDWI    0.012778
          

,feature,importance
1,esa_water_frac_1km,0.155634
0,esa_grass_frac_1km,0.081081
6,gsw_occurrence_mean_1km,0.067563
3,esa_forest_frac_1km,0.065639
5,esa_shrub_frac_1km,0.063249
7,gaia_recent_change_5y_frac,0.056431
15,esa_lccs_class,0.049961
8,gaia_changed_ever_frac_1km,0.045247
4,gsw_recurrence_mean_1km,0.041032
9,esa_cropland_frac_1km,0.038343


In [ ]:
# Predict DRP for validation rows and overwrite DRP column in submission1.csv

# Load submission1.csv (with EC/TA predictions) and validation engineered features
submission_path = "../submission1.csv"
submission = pd.read_csv(submission_path)
val_features = pd.read_csv("../New Datasets/Combined/combined_validation_engineered.csv")

# Standardize keys in submission to match engineered features
sub_std = submission.rename(columns={
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Sample Date": "sample_date",
})

# Join validation features with submission IDs
val_full = val_features.merge(
    sub_std[["latitude", "longitude", "sample_date"]],
    on=["latitude", "longitude", "sample_date"],
    how="inner",
)

print("Validation rows with matched features (DRP):", val_full.shape[0])

# Build X_val using same DRP feature set
X_val = val_full[feature_cols].copy()

drp_pred = final_model.predict(X_val)

# Map predictions back into submission
pred_df = val_full[["latitude", "longitude", "sample_date"]].copy()
pred_df["Dissolved Reactive Phosphorus"] = drp_pred
pred_df = pred_df.rename(columns={
    "latitude": "Latitude",
    "longitude": "Longitude",
    "sample_date": "Sample Date",
})

submission = submission.drop(columns=["Dissolved Reactive Phosphorus"]).merge(
    pred_df,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left",
)

submission.to_csv(submission_path, index=False)
print("Updated submission1.csv with Dissolved Reactive Phosphorus predictions.")
submission.head()


Validation rows with matched features (DRP): 200
Updated submission1.csv with Dissolved Reactive Phosphorus predictions.


,Longitude,Latitude,Sample Date,Electrical Conductance,Total Alkalinity,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,469.42007,152.837880,46.209438
1,26.077500,-33.329167,16-09-2015,161.70978,31.439138,13.076386
2,27.640028,-32.991639,07-05-2015,221.35207,41.791824,18.486393
3,24.439167,-34.096389,07-02-2012,343.66388,44.698444,13.726166
4,28.581667,-32.000556,01-10-2014,530.69940,106.132515,22.970081
